# 🔍 Проверка Cash Flow Statement на всех этапах

**Цель**: Проверить, что Cash Flow Statement правильно собирается на всех этапах:
1. Официальная отчетность
2. RAW БД (после загрузки из Edgar CSV)
3. CANONICAL БД (после normalize_to_canonical)
4. **Иерархическая каноническая структура (CanonicalFinancialStatements)** ⭐
5. Препроцессинг (загрузка исторических данных)
6. Движок модели (ThreeStatementModel)

**Основные проверки**:
- Net Income + Adjustments + WC Changes = CFO
- CFO + CFI + CFF + FX Effect = Net Change
- Cash Opening + Net Change = Cash Ending


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("../../..").resolve()
COMPANY = "us_steel"
croot = ROOT / "companies" / COMPANY

sys.path.insert(0, str(ROOT))

from engine.database.data_mart import get_data_mart
from engine.model.preprocess import ModelPreprocessor
from engine.model3.core import ThreeStatementModel
from engine.model3.io import load_inputs

print(f"✅ Импорты выполнены")
print(f"   ROOT: {ROOT}")
print(f"   COMPANY: {COMPANY}")

## 📊 ШАГ 1: Официальная отчетность (2024)

In [ ]:
# Официальные данные CF 2024
official_cf_2024 = {
    'net_income': 384_000_000,
    'total_da': 913_000_000,
    'asset_impairment_charges_cf': 19_000_000,
    'restructuring_and_other_charges_cf': 8_000_000,
    'loss_on_debt_extinguishment_cf': 2_000_000,
    'pensions_and_other_post_employment_benefits': -133_000_000,
    'active_employee_benefit_investments': 65_000_000,
    'tax_deferred_adjustment': 113_000_000,
    'gain_loss_reversal': -5_000_000,
    'equity_investees_earnings_net_of_distributions': -3_000_000,
    'wc_accounts_receivable_change': 174_000_000,
    'wc_inventory_change': -71_000_000,
    'wc_accounts_payable_change': -285_000_000,
    'wc_income_taxes_receivable_payable_change': -126_000_000,
    'wc_all_other_net': -136_000_000,
    'cfo': 919_000_000,
    'capex': -2287_000_000,
    'disposal_proceeds': 5_000_000,
    'cfi_other': 6_000_000,
    'cfi': -2276_000_000,
    'cff_repayments': -128_000_000,
    'cff_other': -71_000_000,
    'cff': -199_000_000,
    'cf_fx_effect': -19_000_000,
    'net_change': -1575_000_000,
    'cash_opening': 2988_000_000,
    'cash_ending': 1413_000_000,
}

print("="*80)
print("📊 ШАГ 1: ОФИЦИАЛЬНАЯ ОТЧЕТНОСТЬ (2024)")
print("="*80)

net_income_official = official_cf_2024['net_income']
cfo_official = official_cf_2024['cfo']
cfi_official = official_cf_2024['cfi']
cff_official = official_cf_2024['cff']
fx_effect_official = official_cf_2024['cf_fx_effect']
net_change_official = official_cf_2024['net_change']
cash_opening_official = official_cf_2024['cash_opening']
cash_ending_official = official_cf_2024['cash_ending']

# Проверка связок
# CFO = Net Income + Adjustments + WC Changes (упрощенно)
# Net Change = CFO + CFI + CFF + FX Effect
net_change_calc = cfo_official + cfi_official + cff_official + fx_effect_official
diff_net_change = net_change_official - net_change_calc

# Cash Ending = Cash Opening + Net Change
cash_ending_calc = cash_opening_official + net_change_official
diff_cash_ending = cash_ending_official - cash_ending_calc

print(f"\n📋 Cash Flow Statement из официальной отчетности:")
print(f"   Net Income: ${net_income_official:,.0f}")
print(f"   CFO: ${cfo_official:,.0f}")
print(f"   CFI: ${cfi_official:,.0f}")
print(f"   CFF: ${cff_official:,.0f}")
print(f"   FX Effect: ${fx_effect_official:,.0f}")
print(f"   Net Change: ${net_change_official:,.0f}")
print(f"   Cash Opening: ${cash_opening_official:,.0f}")
print(f"   Cash Ending: ${cash_ending_official:,.0f}")

print(f"\n📊 Проверка связок:")
print(f"   CFO + CFI + CFF + FX Effect = Net Change: ${cfo_official:,.0f} + ${cfi_official:,.0f} + ${cff_official:,.0f} + ${fx_effect_official:,.0f} = ${net_change_calc:,.0f} (ожидается ${net_change_official:,.0f}, diff: ${diff_net_change:,.0f})")
print(f"   Cash Opening + Net Change = Cash Ending: ${cash_opening_official:,.0f} + ${net_change_official:,.0f} = ${cash_ending_calc:,.0f} (ожидается ${cash_ending_official:,.0f}, diff: ${diff_cash_ending:,.0f})")

all_ok = abs(diff_net_change) < 1e-6 and abs(diff_cash_ending) < 1e-6
if all_ok:
    print(f"\n✅ Все связки сходятся!")
else:
    print(f"\n⚠️ Есть расхождения в связках!")

## 📊 ШАГ 2: RAW БД (после загрузки из Edgar CSV)

In [ ]:
mart = get_data_mart(ROOT, COMPANY)
cf_raw = mart.get_history_cash_flow(canonical=False)

print("="*80)
print("📊 ШАГ 2: RAW БД (после загрузки из Edgar CSV)")
print("="*80)

def get_raw_value(metric_name):
    if cf_raw.empty:
        return None
    metric_row = cf_raw[cf_raw['metric'].str.lower() == metric_name.lower()]
    if metric_row.empty:
        return None
    if year_str not in metric_row.columns:
        return None
    return pd.to_numeric(metric_row.iloc[0][year_str], errors='coerce')

cfo_raw = get_raw_value('cfo') or get_raw_value('net_cash_provided_by_operating_activities')
cfi_raw = get_raw_value('cfi') or get_raw_value('net_cash_used_in_investing_activities')
cff_raw = get_raw_value('cff') or get_raw_value('net_cash_used_in_financing_activities')
fx_effect_raw = get_raw_value('cf_fx_effect') or get_raw_value('effect_of_exchange_rate_changes')
net_change_raw = get_raw_value('net_change') or get_raw_value('net_change_in_cash')
cash_opening_raw = get_raw_value('cash_opening') or get_raw_value('cash_at_beginning_of_year')
cash_ending_raw = get_raw_value('cash_ending') or get_raw_value('cash_at_end_of_year')

print(f"\n📋 Cash Flow Statement из RAW БД:")
print(f"   CFO: ${cfo_raw:,.0f}" if pd.notna(cfo_raw) else "   CFO: N/A")
print(f"   CFI: ${cfi_raw:,.0f}" if pd.notna(cfi_raw) else "   CFI: N/A")
print(f"   CFF: ${cff_raw:,.0f}" if pd.notna(cff_raw) else "   CFF: N/A")
print(f"   FX Effect: ${fx_effect_raw:,.0f}" if pd.notna(fx_effect_raw) else "   FX Effect: N/A")
print(f"   Net Change: ${net_change_raw:,.0f}" if pd.notna(net_change_raw) else "   Net Change: N/A")
print(f"   Cash Opening: ${cash_opening_raw:,.0f}" if pd.notna(cash_opening_raw) else "   Cash Opening: N/A")
print(f"   Cash Ending: ${cash_ending_raw:,.0f}" if pd.notna(cash_ending_raw) else "   Cash Ending: N/A")

if pd.notna(cfo_raw) and pd.notna(cfi_raw) and pd.notna(cff_raw) and pd.notna(fx_effect_raw):
    net_change_calc_raw = cfo_raw + cfi_raw + cff_raw + fx_effect_raw
    diff_net_change_raw = net_change_raw - net_change_calc_raw if pd.notna(net_change_raw) else None
    if diff_net_change_raw is not None:
        print(f"\n📊 Проверка связок:")
        print(f"   CFO + CFI + CFF + FX Effect = Net Change: diff = ${diff_net_change_raw:,.0f}")
        if abs(diff_net_change_raw) < 1e-6:
            print(f"   ✅ Связка CFO + CFI + CFF + FX Effect = Net Change сходится!")
        else:
            print(f"   ⚠️ Связка CFO + CFI + CFF + FX Effect = Net Change НЕ сходится!")

mart.close()

## 📊 ШАГ 3: CANONICAL БД (после normalize_to_canonical)

In [ ]:
cf_canonical = mart.get_history_cash_flow(canonical=True)

print("="*80)
print("📊 ШАГ 3: CANONICAL БД (после normalize_to_canonical)")
print("="*80)

def get_canonical_value(metric_name):
    if cf_canonical.empty:
        return None
    metric_row = cf_canonical[cf_canonical['metric'].str.lower() == metric_name.lower()]
    if metric_row.empty:
        return None
    if year_str not in metric_row.columns:
        return None
    return pd.to_numeric(metric_row.iloc[0][year_str], errors='coerce')

cfo_canonical = get_canonical_value('cfo')
cfi_canonical = get_canonical_value('cfi')
cff_canonical = get_canonical_value('cff')
fx_effect_canonical = get_canonical_value('cf_fx_effect')
net_change_canonical = get_canonical_value('net_change')
cash_opening_canonical = get_canonical_value('cash_opening')
cash_ending_canonical = get_canonical_value('cash_ending')

if pd.notna(cfo_canonical):
    print(f"\n📋 Cash Flow Statement из CANONICAL БД:")
    print(f"   CFO: ${cfo_canonical:,.0f}")
    print(f"   CFI: ${cfi_canonical:,.0f}" if pd.notna(cfi_canonical) else "   CFI: N/A")
    print(f"   CFF: ${cff_canonical:,.0f}" if pd.notna(cff_canonical) else "   CFF: N/A")
    print(f"   FX Effect: ${fx_effect_canonical:,.0f}" if pd.notna(fx_effect_canonical) else "   FX Effect: N/A")
    print(f"   Net Change: ${net_change_canonical:,.0f}" if pd.notna(net_change_canonical) else "   Net Change: N/A")
    print(f"   Cash Opening: ${cash_opening_canonical:,.0f}" if pd.notna(cash_opening_canonical) else "   Cash Opening: N/A")
    print(f"   Cash Ending: ${cash_ending_canonical:,.0f}" if pd.notna(cash_ending_canonical) else "   Cash Ending: N/A")

    if pd.notna(cfi_canonical) and pd.notna(cff_canonical) and pd.notna(fx_effect_canonical):
        net_change_calc_canonical = cfo_canonical + cfi_canonical + cff_canonical + fx_effect_canonical
        diff_net_change_canonical = net_change_canonical - net_change_calc_canonical if pd.notna(net_change_canonical) else None
        if diff_net_change_canonical is not None:
            print(f"\n📊 Проверка связок:")
            print(f"   CFO + CFI + CFF + FX Effect = Net Change: diff = ${diff_net_change_canonical:,.0f}")
            if abs(diff_net_change_canonical) < 1e-6:
                print(f"   ✅ Связка CFO + CFI + CFF + FX Effect = Net Change сходится!")
            else:
                print(f"   ⚠️ Связка CFO + CFI + CFF + FX Effect = Net Change НЕ сходится!")

    cash_ending_calc_canonical = cash_opening_canonical + net_change_canonical if pd.notna(cash_opening_canonical) and pd.notna(net_change_canonical) else None
    if cash_ending_calc_canonical is not None:
        diff_cash_ending_canonical = cash_ending_canonical - cash_ending_calc_canonical if pd.notna(cash_ending_canonical) else None
        if diff_cash_ending_canonical is not None:
            print(f"   Cash Opening + Net Change = Cash Ending: diff = ${diff_cash_ending_canonical:,.0f}")
            if abs(diff_cash_ending_canonical) < 1e-6:
                print(f"   ✅ Связка Cash Opening + Net Change = Cash Ending сходится!")
            else:
                print(f"   ⚠️ Связка Cash Opening + Net Change = Cash Ending НЕ сходится!")

    print(f"\n📊 Сравнение с RAW БД:")
    if pd.notna(cfo_raw):
        print(f"   CFO: ${cfo_canonical:,.0f} vs ${cfo_raw:,.0f} (diff: ${cfo_canonical - cfo_raw:,.0f})")
else:
    print("⚠️ Не все ключевые метрики найдены в CANONICAL БД")

## 📊 ШАГ 3.5: Иерархическая каноническая структура (CanonicalFinancialStatements) ⭐


In [ ]:
# Загружаем иерархическую каноническую структуру
print("="*80)
print("📊 ШАГ 3.5: ИЕРАРХИЧЕСКАЯ КАНОНИЧЕСКАЯ СТРУКТУРА (CanonicalFinancialStatements)")
print("="*80)

canonical_hierarchical = mart.get_canonical_financial_statements(year=year)

if canonical_hierarchical is None:
    print("⚠️ Иерархическая каноническая структура недоступна (адаптеры не найдены или данные отсутствуют)")
    cfo_hierarchical = None
    cfi_hierarchical = None
    cff_hierarchical = None
    net_change_hierarchical = None
    cash_ending_hierarchical = None
else:
    # Получаем значения из иерархической структуры
    cf_hierarchical = canonical_hierarchical.cash_flow
    
    # Получаем ключевые метрики
    cfo_hierarchical = cf_hierarchical.cfo.cfo_total
    cfi_hierarchical = cf_hierarchical.cfi.cfi_total
    cff_hierarchical = cf_hierarchical.cff.cff_total
    fx_effect_hierarchical = cf_hierarchical.cf_fx_effect if hasattr(cf_hierarchical, 'cf_fx_effect') else None
    net_change_hierarchical = cf_hierarchical.net_change_in_cash
    cash_opening_hierarchical = cf_hierarchical.cash_beginning
    cash_ending_hierarchical = cf_hierarchical.cash_ending
    
    # Проверка связок
    net_change_calc_hierarchical = cfo_hierarchical + cfi_hierarchical + cff_hierarchical + (fx_effect_hierarchical or 0) if pd.notna(cfo_hierarchical) and pd.notna(cfi_hierarchical) and pd.notna(cff_hierarchical) else None
    cash_ending_calc_hierarchical = cash_opening_hierarchical + net_change_hierarchical if pd.notna(cash_opening_hierarchical) and pd.notna(net_change_hierarchical) else None
    
    print(f"\n📋 Cash Flow Statement из иерархической канонической структуры:")
    print(f"   CFO: ${cfo_hierarchical:,.0f}" if pd.notna(cfo_hierarchical) else "   CFO: N/A")
    print(f"   CFI: ${cfi_hierarchical:,.0f}" if pd.notna(cfi_hierarchical) else "   CFI: N/A")
    print(f"   CFF: ${cff_hierarchical:,.0f}" if pd.notna(cff_hierarchical) else "   CFF: N/A")
    print(f"   FX Effect: ${fx_effect_hierarchical:,.0f}" if pd.notna(fx_effect_hierarchical) else "   FX Effect: N/A")
    print(f"   Net Change: ${net_change_hierarchical:,.0f}" if pd.notna(net_change_hierarchical) else "   Net Change: N/A")
    print(f"   Cash Opening: ${cash_opening_hierarchical:,.0f}" if pd.notna(cash_opening_hierarchical) else "   Cash Opening: N/A")
    print(f"   Cash Ending: ${cash_ending_hierarchical:,.0f}" if pd.notna(cash_ending_hierarchical) else "   Cash Ending: N/A")
    
    if pd.notna(net_change_calc_hierarchical):
        diff_net_change_hierarchical = net_change_hierarchical - net_change_calc_hierarchical if pd.notna(net_change_hierarchical) else None
        if diff_net_change_hierarchical is not None:
            print(f"\n📊 Проверка связок:")
            print(f"   CFO + CFI + CFF + FX Effect = Net Change: diff = ${diff_net_change_hierarchical:,.0f}")
            if abs(diff_net_change_hierarchical) < 1e-6:
                print(f"   ✅ Связка CFO + CFI + CFF + FX Effect = Net Change сходится!")
            else:
                print(f"   ⚠️ Связка CFO + CFI + CFF + FX Effect = Net Change НЕ сходится!")
    
    if pd.notna(cash_ending_calc_hierarchical):
        diff_cash_ending_hierarchical = cash_ending_hierarchical - cash_ending_calc_hierarchical if pd.notna(cash_ending_hierarchical) else None
        if diff_cash_ending_hierarchical is not None:
            print(f"   Cash Opening + Net Change = Cash Ending: diff = ${diff_cash_ending_hierarchical:,.0f}")
            if abs(diff_cash_ending_hierarchical) < 1e-6:
                print(f"   ✅ Связка Cash Opening + Net Change = Cash Ending сходится!")
            else:
                print(f"   ⚠️ Связка Cash Opening + Net Change = Cash Ending НЕ сходится!")
    
    # Сравнение с CANONICAL БД (плоский формат)
    if pd.notna(cfo_canonical):
        print(f"\n📊 Сравнение с CANONICAL БД (плоский формат):")
        print(f"   CFO: ${cfo_hierarchical:,.0f} vs ${cfo_canonical:,.0f} (diff: ${cfo_hierarchical - cfo_canonical:,.0f})")
        if pd.notna(net_change_canonical):
            print(f"   Net Change: ${net_change_hierarchical:,.0f} vs ${net_change_canonical:,.0f} (diff: ${net_change_hierarchical - net_change_canonical:,.0f})")
    
    # Проверка детализации CFO (если доступна)
    cfo_detail = cf_hierarchical.cfo
    if hasattr(cfo_detail, 'change_ar') and cfo_detail.change_ar is not None:
        print(f"\n💡 Детализация CFO:")
        print(f"   Change AR: ${cfo_detail.change_ar:,.0f}" if pd.notna(cfo_detail.change_ar) else "   Change AR: N/A")
        print(f"   Change Inventory: ${cfo_detail.change_inventory:,.0f}" if pd.notna(cfo_detail.change_inventory) else "   Change Inventory: N/A")
        print(f"   Change AP: ${cfo_detail.change_ap:,.0f}" if pd.notna(cfo_detail.change_ap) else "   Change AP: N/A")
        if hasattr(cfo_detail, 'lease_payments_cfo_operating') and cfo_detail.lease_payments_cfo_operating is not None:
            print(f"   Operating Lease Payments (CFO): ${cfo_detail.lease_payments_cfo_operating:,.0f}")
    
    # Проверка детализации CFF (если доступна)
    cff_detail = cf_hierarchical.cff
    if hasattr(cff_detail, 'finance_lease_principal') and cff_detail.finance_lease_principal is not None:
        print(f"\n💡 Детализация CFF:")
        print(f"   Finance Lease Principal: ${cff_detail.finance_lease_principal:,.0f}")
        print(f"   Debt Issuance: ${cff_detail.debt_issuance:,.0f}" if pd.notna(cff_detail.debt_issuance) else "   Debt Issuance: N/A")
        print(f"   Debt Repayment: ${cff_detail.debt_repayment:,.0f}" if pd.notna(cff_detail.debt_repayment) else "   Debt Repayment: N/A")
        print(f"   Dividends Paid: ${cff_detail.dividends_paid:,.0f}" if pd.notna(cff_detail.dividends_paid) else "   Dividends Paid: N/A")


## 📊 ШАГ 4: Препроцессинг (загрузка исторических данных)

In [ ]:
preprocessor = ModelPreprocessor(ROOT, COMPANY)
hist_cf = preprocessor.mart.get_history_cash_flow(canonical=True)

print("="*80)
print("📊 ШАГ 4: ПРЕПРОЦЕССИНГ (загрузка исторических данных)")
print("="*80)

def get_hist_value(metric_name):
    if hist_cf.empty:
        return None
    metric_row = hist_cf[hist_cf['metric'].str.lower() == metric_name.lower()]
    if metric_row.empty:
        return None
    if year_str not in metric_row.columns:
        return None
    return pd.to_numeric(metric_row.iloc[0][year_str], errors='coerce')

cfo_preprocess = get_hist_value('cfo')
cfi_preprocess = get_hist_value('cfi')
cff_preprocess = get_hist_value('cff')
fx_effect_preprocess = get_hist_value('cf_fx_effect')
net_change_preprocess = get_hist_value('net_change')
cash_opening_preprocess = get_hist_value('cash_opening')
cash_ending_preprocess = get_hist_value('cash_ending')

if pd.notna(cfo_preprocess):
    print(f"\n📋 Cash Flow Statement из исторических данных:")
    print(f"   CFO: ${cfo_preprocess:,.0f}")
    print(f"   CFI: ${cfi_preprocess:,.0f}" if pd.notna(cfi_preprocess) else "   CFI: N/A")
    print(f"   CFF: ${cff_preprocess:,.0f}" if pd.notna(cff_preprocess) else "   CFF: N/A")
    print(f"   FX Effect: ${fx_effect_preprocess:,.0f}" if pd.notna(fx_effect_preprocess) else "   FX Effect: N/A")
    print(f"   Net Change: ${net_change_preprocess:,.0f}" if pd.notna(net_change_preprocess) else "   Net Change: N/A")
    print(f"   Cash Opening: ${cash_opening_preprocess:,.0f}" if pd.notna(cash_opening_preprocess) else "   Cash Opening: N/A")
    print(f"   Cash Ending: ${cash_ending_preprocess:,.0f}" if pd.notna(cash_ending_preprocess) else "   Cash Ending: N/A")

    if pd.notna(cfi_preprocess) and pd.notna(cff_preprocess) and pd.notna(fx_effect_preprocess):
        net_change_calc_preprocess = cfo_preprocess + cfi_preprocess + cff_preprocess + fx_effect_preprocess
        diff_net_change_preprocess = net_change_preprocess - net_change_calc_preprocess if pd.notna(net_change_preprocess) else None
        if diff_net_change_preprocess is not None:
            print(f"\n📊 Проверка связок:")
            print(f"   CFO + CFI + CFF + FX Effect = Net Change: diff = ${diff_net_change_preprocess:,.0f}")
            if abs(diff_net_change_preprocess) < 1e-6:
                print(f"   ✅ Связка CFO + CFI + CFF + FX Effect = Net Change сходится!")
            else:
                print(f"   ⚠️ Связка CFO + CFI + CFF + FX Effect = Net Change НЕ сходится!")

    if pd.notna(cash_opening_canonical):
        print(f"\n📊 Сравнение с CANONICAL БД:")
        print(f"   CFO: ${cfo_preprocess:,.0f} vs ${cfo_canonical:,.0f} (diff: ${cfo_preprocess - cfo_canonical:,.0f})")
else:
    print("⚠️ Не все ключевые метрики найдены в исторических данных")

## 📊 ШАГ 5: Движок модели (ThreeStatementModel)

In [ ]:
hist_state, history, drivers, canonical = load_inputs(root=ROOT, company=COMPANY)model = ThreeStatementModel(**inputs)model.build()print("="*80)print("📊 ШАГ 5: ДВИЖОК МОДЕЛИ (ThreeStatementModel)")print("="*80)cf_2024 = model.CF[2024]cfo_model = cf_2024.cfocfi_model = cf_2024.cficff_model = cf_2024.cfffx_effect_model = cf_2024.cf_fx_effectnet_change_model = cf_2024.net_changecash_opening_model = cf_2024.cash_openingcash_ending_model = cf_2024.cash_endingnet_change_calc_model = cfo_model + cfi_model + cff_model + fx_effect_modelcash_ending_calc_model = cash_opening_model + net_change_modeldiff_net_change_model = net_change_model - net_change_calc_modeldiff_cash_ending_model = cash_ending_model - cash_ending_calc_modelprint(f"\n📋 Cash Flow Statement из модели (2024 год):")print(f"   CFO: ${cfo_model:,.0f}")print(f"   CFI: ${cfi_model:,.0f}")print(f"   CFF: ${cff_model:,.0f}")print(f"   FX Effect: ${fx_effect_model:,.0f}")print(f"   Net Change: ${net_change_model:,.0f}")print(f"   Cash Opening: ${cash_opening_model:,.0f}")print(f"   Cash Ending: ${cash_ending_model:,.0f}")print(f"\n📊 Проверка связок:")print(f"   CFO + CFI + CFF + FX Effect = Net Change: diff = ${diff_net_change_model:,.0f}")if abs(diff_net_change_model) < 1e-6:    print(f"   ✅ Связка CFO + CFI + CFF + FX Effect = Net Change сходится!")else:    print(f"   ⚠️ Связка CFO + CFI + CFF + FX Effect = Net Change НЕ сходится!")print(f"   Cash Opening + Net Change = Cash Ending: diff = ${diff_cash_ending_model:,.0f}")if abs(diff_cash_ending_model) < 1e-6:    print(f"   ✅ Связка Cash Opening + Net Change = Cash Ending сходится!")else:    print(f"   ⚠️ Связка Cash Opening + Net Change = Cash Ending НЕ сходится!")if pd.notna(cfo_preprocess):    print(f"\n📊 Сравнение с препроцессингом:")    print(f"   CFO: ${cfo_model:,.0f} vs ${cfo_preprocess:,.0f} (diff: ${cfo_model - cfo_preprocess:,.0f})")    print(f"   Net Change: ${net_change_model:,.0f} vs ${net_change_preprocess:,.0f} (diff: ${net_change_model - net_change_preprocess:,.0f})")

## 📊 ИТОГОВАЯ СВОДКА

In [ ]:
from IPython.display import display, Markdown

print("="*80)
print("📊 ИТОГОВАЯ СВОДКА ПРОВЕРКИ CASH FLOW STATEMENT")
print("="*80)

summary_data = []

summary_data.append({
    'Этап': 'Официальная отчетность',
    'CFO': f"${cfo_official:,.0f}",
    'CFI': f"${cfi_official:,.0f}",
    'CFF': f"${cff_official:,.0f}",
    'Net Change': f"${net_change_official:,.0f}",
    'Cash Ending': f"${cash_ending_official:,.0f}",
    'Статус': '✅' if all_ok else '⚠️'
})

if pd.notna(cfo_raw):
    summary_data.append({
        'Этап': 'RAW БД',
        'CFO': f"${cfo_raw:,.0f}" if pd.notna(cfo_raw) else 'N/A',
        'CFI': f"${cfi_raw:,.0f}" if pd.notna(cfi_raw) else 'N/A',
        'CFF': f"${cff_raw:,.0f}" if pd.notna(cff_raw) else 'N/A',
        'Net Change': f"${net_change_raw:,.0f}" if pd.notna(net_change_raw) else 'N/A',
        'Cash Ending': f"${cash_ending_raw:,.0f}" if pd.notna(cash_ending_raw) else 'N/A',
        'Статус': '✅'
    })

if pd.notna(cfo_canonical):
    summary_data.append({
        'Этап': 'CANONICAL БД',
        'CFO': f"${cfo_canonical:,.0f}",
        'CFI': f"${cfi_canonical:,.0f}" if pd.notna(cfi_canonical) else 'N/A',
        'CFF': f"${cff_canonical:,.0f}" if pd.notna(cff_canonical) else 'N/A',
        'Net Change': f"${net_change_canonical:,.0f}" if pd.notna(net_change_canonical) else 'N/A',
        'Cash Ending': f"${cash_ending_canonical:,.0f}" if pd.notna(cash_ending_canonical) else 'N/A',
        'Статус': '✅'
    })

if pd.notna(cfo_preprocess):
    summary_data.append({
        'Этап': 'Препроцессинг',
        'CFO': f"${cfo_preprocess:,.0f}",
        'CFI': f"${cfi_preprocess:,.0f}" if pd.notna(cfi_preprocess) else 'N/A',
        'CFF': f"${cff_preprocess:,.0f}" if pd.notna(cff_preprocess) else 'N/A',
        'Net Change': f"${net_change_preprocess:,.0f}" if pd.notna(net_change_preprocess) else 'N/A',
        'Cash Ending': f"${cash_ending_preprocess:,.0f}" if pd.notna(cash_ending_preprocess) else 'N/A',
        'Статус': '✅'
    })

summary_data.append({
    'Этап': 'Движок модели',
    'CFO': f"${cfo_model:,.0f}",
    'CFI': f"${cfi_model:,.0f}",
    'CFF': f"${cff_model:,.0f}",
    'Net Change': f"${net_change_model:,.0f}",
    'Cash Ending': f"${cash_ending_model:,.0f}",
    'Статус': '✅' if abs(diff_net_change_model) < 1e-6 and abs(diff_cash_ending_model) < 1e-6 else '⚠️'
})

summary_df = pd.DataFrame(summary_data)
display(Markdown("#### 📊 Сводная таблица проверки Cash Flow Statement на всех этапах:"))
display(summary_df.style.set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '5px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
]).set_properties(**{'font-size': '9px'}))

print("\n" + "="*80)
print("✅ ПРОВЕРКА ЗАВЕРШЕНА")
print("="*80)
print("\n💡 Интерпретация результатов:")
print("   ✅ - Связки сходятся (CFO + CFI + CFF + FX Effect = Net Change, Cash Opening + Net Change = Cash Ending)")
print("   ⚠️ - Есть расхождения в связках")
print("\n   Если связки не сходятся на каком-то этапе, проверьте:")
print("   - Правильность маппинга метрик")
print("   - Правильность суммирования статей")
print("   - Потерю данных при преобразовании")
print("   - Логику препроцессинга или движка")